# <font color='blue'> Chapter 58: Prioritized Experience Replay (PER) </font>

Experience Replay was one of the major innovations introduced by Deep Q Networks.

Instead of learning immediately from each interaction,

the agent stores experiences inside a replay buffer and later samples them randomly for training.

Although this stabilizes learning,

uniform random sampling is inefficient.

Some experiences teach the agent far more than others.

For example,

unexpected rewards,

rare events,

or transitions with large prediction errors often contain much more useful learning information.

Prioritized Experience Replay (PER) improves sample efficiency by replaying these important experiences more frequently.

---



# <font color='orange'> 1. Motivation </font>

Imagine studying for an examination.

You solve

100 problems.

Most are easy,

but a few reveal mistakes that you repeatedly make.

Which problems should you review?

```
Easy Problem

↓

Already Understood

Hard Problem

↓

Large Mistake

↓

Review Again
```

Naturally,

you spend more time reviewing the difficult questions.

PER applies exactly the same principle.

---



# <font color='orange'> 2. Standard Experience Replay </font>

The replay buffer stores transitions

$$
(s,a,r,s').
$$

Training samples are selected

uniformly at random.

```
Replay Buffer

↓

Random Sample

↓

Gradient Update
```

Every experience has exactly the same probability of being selected.

---



# <font color='orange'> 3. Why Uniform Sampling is Inefficient </font>

Suppose

the replay buffer contains

```
Transition 1

Transition 2

Transition 3

...

Transition 1,000,000
```

Some transitions

have already been learned well.

Others still contain large prediction errors.

Uniform sampling

treats both equally,

even though the second group contains much more useful information.

---



# <font color='orange'> 4. Temporal Difference Error </font>

The importance of an experience is measured using

the Temporal Difference (TD) error.

Recall

$$
\boxed{
\delta
=
r
+
\gamma
\max_a
Q(s',a)
-
Q(s,a).
}
$$

Interpretation

- Small TD error

↓

the prediction is already accurate.

- Large TD error

↓

the network still has much to learn.

---



# <font color='orange'> 5. Priority </font>

Each transition receives

a priority

based on the magnitude of its TD error.

$$
\boxed{
p_i
=
|\delta_i|
+
\varepsilon,
}
$$

where

- \(\delta_i\) is the TD error,
- \(\varepsilon>0\) is a small constant ensuring that every transition has a non-zero priority.

Larger prediction errors therefore receive higher priority.

---



# <font color='orange'> 6. Sampling Probability </font>

Transitions are no longer sampled uniformly.

Instead,

their probability is

$$
\boxed{
P(i)
=
\frac{p_i^\alpha}
{\sum_k p_k^\alpha},
}
$$

where

- \(p_i\) is the priority,
- \(\alpha\in[0,1]\) controls the strength of prioritization.

Special cases

- \(\alpha=0\)

↓

uniform random sampling.

- \(\alpha=1\)

↓

fully prioritized replay.

Typical implementations use intermediate values,

such as

$$
\alpha
=
0.6.
$$

---



# <font color='orange'> 7. Why Prioritization Introduces Bias </font>

Suppose

high-error transitions

are sampled much more frequently.

The mini-batches

no longer represent

the true experience distribution.

This introduces

sampling bias.

Without correction,

the learned value function may become systematically distorted.

---



# <font color='orange'> 8. Importance Sampling </font>

To compensate for this bias,

PER introduces

**Importance Sampling (IS) weights**.

Each sampled transition receives

the weight

$$
\boxed{
w_i
=
\left(
\frac{1}
{N\,P(i)}
\right)^\beta,
}
$$

where

- \(N\) is the replay-buffer size,
- \(P(i)\) is the sampling probability,
- \(\beta\in[0,1]\) controls the amount of bias correction.

Transitions sampled very frequently receive smaller weights,

while rare transitions receive larger weights.

---



# <font color='orange'> 9. Training Pipeline </font>

A typical PER iteration proceeds as follows.

```
Collect Experience

↓

Replay Buffer

↓

Compute TD Errors

↓

Assign Priorities

↓

Sample According to Priority

↓

Compute Importance Weights

↓

Gradient Update

↓

Update Priorities
```

Notice that priorities change during training,

because the TD errors also change.

---



# <font color='orange'> 10. Updating Priorities </font>

After each gradient update,

new TD errors are computed.

The priorities are then updated,

allowing the replay buffer to focus continuously on the experiences that remain difficult for the network.

Thus,

the replay distribution evolves together with the learning process.

---



# <font color='orange'> 11. Advantages </font>

Prioritized Experience Replay

- improves sample efficiency,
- accelerates learning,
- focuses computation on informative experiences,
- often reaches higher performance using fewer interactions with the environment.

These benefits are especially important when interactions with the environment are expensive.

---



# <font color='orange'> 12. Limitations </font>

PER also introduces additional complexity.

It requires

- storing priorities,
- updating priorities after training,
- computing importance weights,
- maintaining efficient data structures for sampling.

Furthermore,

incorrect prioritization may occasionally overemphasize noisy transitions.

---



# <font color='orange'> 13. Applications </font>

PER has been successfully applied to

- Atari game playing,
- robotic manipulation,
- autonomous driving,
- industrial scheduling,
- resource allocation.

It is widely used as a standard enhancement for value-based Deep Reinforcement Learning algorithms.

---



# <font color='orange'> 14. Relationship to Rainbow DQN </font>

Rainbow DQN combines several complementary improvements,

including

- Double DQN,
- Dueling Networks,
- Prioritized Experience Replay,
- Multi-step Learning,
- Distributional Reinforcement Learning,
- Noisy Networks.

PER contributes by improving

how training data are selected,

rather than changing the neural network or the learning rule.

---

# <font color='red'> 15. Mathematical Foundations </font>

Temporal Difference Error

$$
\boxed{
\delta_i
=
r_i
+
\gamma
\max_a
Q(s_i',a)
-
Q(s_i,a_i).
}
$$

Priority

$$
\boxed{
p_i
=
|\delta_i|
+
\varepsilon.
}
$$

Sampling Probability

$$
\boxed{
P(i)
=
\frac{p_i^\alpha}
{\sum_k p_k^\alpha}.
}
$$

Importance Sampling Weight

$$
\boxed{
w_i
=
\left(
\frac{1}
{N\,P(i)}
\right)^\beta.
}
$$

The weighted loss for a sampled transition is commonly written as

$$
\boxed{
L_i
=
w_i
\left(
y_i
-
Q(s_i,a_i)
\right)^2,
}
$$

where

$$
y_i
$$

is the target value (for example, from DQN or Double DQN).

The importance sampling weight reduces the bias introduced by prioritized sampling.

---



# <font color='orange'> 16. Uniform Replay vs Prioritized Replay </font>

| Uniform Replay | Prioritized Experience Replay |
|:---|:---|
| Equal probability for every transition | Higher probability for informative transitions |
| Simpler implementation | Requires maintaining priorities |
| No sampling bias | Introduces bias that is corrected using importance sampling |
| Lower sample efficiency | Higher sample efficiency |

---



# <font color='orange'> 17. Common Misconceptions </font>

### Misconception 1

> PER always samples the transition with the largest TD error.

**False.**

PER samples **probabilistically**. Higher-priority transitions are more likely to be selected, but lower-priority transitions still have a chance of being sampled.

---

### Misconception 2

> A large TD error always means the transition is important.

**Not necessarily.**

A large TD error may indicate that the transition is informative, but it can also arise from noisy rewards, stochastic environments, or approximation errors. PER assumes that larger TD errors are generally more useful for learning, but this heuristic is not perfect.

---

### Misconception 3

> Importance Sampling removes all bias.

**False.**

Importance Sampling compensates for the change in sampling distribution, reducing the bias introduced by prioritized replay. In practice, the correction is often annealed by gradually increasing \(\beta\) toward 1 during training, so some bias may remain early in learning.

---

# <font color='purple'> 18. Conceptual Summary </font>

| Concept | Description |
|:---|:---|
| Experience Replay | Stores transitions for later training |
| Temporal Difference Error | Measures the prediction error for a transition |
| Priority | Importance assigned to a transition |
| Prioritized Experience Replay | Samples informative transitions more frequently |
| Sampling Probability | Probability of selecting a transition based on its priority |
| Importance Sampling | Corrects for the bias introduced by non-uniform sampling |
| Sample Efficiency | Amount learned from each interaction with the environment |
| Rainbow DQN | Combines PER with several other improvements to DQN |

> **Key Insight:** Prioritized Experience Replay improves Deep Q Networks by replaying the experiences that are expected to contribute most to learning. Instead of treating every transition equally, PER assigns higher sampling probabilities to transitions with larger temporal difference errors, allowing the agent to focus on experiences where its predictions are least accurate. Importance Sampling weights compensate for the resulting sampling bias, making PER a powerful and widely used technique for improving the efficiency of value-based deep reinforcement learning.